In [ ]:
# system utilities
import os
import glob

# modules to query skymapper 
import urllib.request

# plotting module
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# numeric and table data modules
import numpy as np
from numpy import errstate,isneginf,array

# dataframes module
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# module to read Fits files
from astropy.io import fits

# image visualization and World Coordinate System modules
from astropy.visualization import simple_norm
from astropy.wcs import WCS

# units and coordinate handling
from astropy import units as u
from astropy.coordinates import SkyCoord

# Apperture photometry module
from photutils import SkyCircularAperture, aperture_photometry




In [ ]:
def getKeyListVal(filename, keys):
    keys_dict = {}
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        for key in keys:
            keys_dict[key] = primary_header[key]
        return keys_dict

In [ ]:
reduced_dir = '/home/fselman/omegacam/fernando/reduced/science/J1649-3307_164955p600_m330702p0/r_SDSS/2017-04-13T05:11:38/'
current_dir = '/home/fselman/omegacam/fernando/'
print('current directory:', os.getcwd())
os.chdir(current_dir)

coord = getKeyListVal(reduced_dir+'coadd.fits', ['CRVAL1', 'CRVAL2'])
ra_str = ("%-10.5f" % coord['CRVAL1']).strip()
de_str = ("%-10.5f" % coord['CRVAL2']).strip()
radius = '0.7' 
query_str = "http://skymapper.anu.edu.au/sm-cone/public/query?RA="+ra_str+"&DEC="+de_str+"&SR=0.80&VERB=3&RESPONSEFORMAT=CSV"
print(query_str)
with urllib.request.urlopen(query_str) as f:
    charset = f.headers.get_content_charset()
    # print(charset)
    df_cat = pd.DataFrame([x.split(',') for x in f.read().decode(charset).split('\r\n')])

# we set column 1 as the header collumn
df_cat.columns = df_cat.iloc[0]

# we select only relevant collumns
df_cat = df_cat[['object_id', 'raj2000', 'dej2000', 'class_star', 'g_psf', 'e_g_psf', 'r_psf', 'e_r_psf']][1:]



In [ ]:
# we convert to numbers where appropriate
df_cat['raj2000']=df_cat.raj2000.replace('',np.nan).astype(float)
df_cat['raj2000'] = df_cat['raj2000'].astype('float64')

df_cat['dej2000']=df_cat.dej2000.replace('',np.nan).astype(float)
df_cat['dej2000'] = df_cat['dej2000'].astype('float64')

df_cat['g_psf'] = df_cat.g_psf.replace('', np.nan).astype('float64')
df_cat['e_g_psf'] = df_cat.e_g_psf.replace('', np.nan).astype('float64')
df_cat['r_psf'] = df_cat.r_psf.replace('', np.nan).astype('float64')
df_cat['e_r_psf'] = df_cat.e_r_psf.replace('', np.nan).astype('float64')

mlim = 17.0
df_cat = df_cat[(df_cat['r_psf'] < mlim) & \
                (df_cat['g_psf'] < mlim) & \
                (df_cat['raj2000'] == df_cat['raj2000']) & \
                (df_cat['dej2000'] == df_cat['dej2000'])]
# df_cat

In [ ]:
# Sanity check: the apertures are on top of the stars...

ra = np.array(df_cat['raj2000'])*u.deg
dec = np.array(df_cat['dej2000'])*u.deg

# filter bad coordinates
ra_bad =  np.where(np.isnan(ra))

# Define positions and apertures
positions = SkyCoord(ra=ra, dec=dec,frame='icrs')
aperture = SkyCircularAperture(positions, r=2. * u.arcsec)

# open image and plot apertures
data, hdu = fits.getdata(reduced_dir+'coadd.fits', header=True)
# abd we plot
# read wcs from image keywords
wcs = WCS(reduced_dir+'coadd.fits')  

# data = fits.getdata(reduced_dir+'coadd.fits')
#data = hdu[0].data

# we do the aperture photometry -- first good enough version.
# For a better approach we should do curve of growth..
#
phot_table = aperture_photometry(data, aperture, wcs=wcs)
phot_table['aperture_sum'].info.format = '%.8g'  # for consistent table output

# we read the magnitudes from the tables
ocam_r_SDSS = -2.5*np.log10(np.array(phot_table['aperture_sum']))




In [ ]:
# we plot the magnitudes from the photometry table
sm_r_SDSS = np.array(df_cat['r_psf'])
sm_g_SDSS = np.array(df_cat['g_psf'])

# but first we determine zero point and slope
# of magnitude curve.
def getZP(sm_r, sm_g, ocam_r, mag_min, mag_max, dmag_filter):
    x = sm_r
    xg = sm_g
    y = ocam_r

    X = []
    Xg = []
    Y = []
    for i in range(len(x)):
        if (x[i] == x[i]) and \
            not (np.isinf(y[i]) or np.isnan(y[i])) and \
            (x[i] > mag_min and x[i] < mag_max):
            X.append(x[i])
            Xg.append(xg[i])
            Y.append(y[i])

    X = np.array(X)
    Xg = np.array(Xg)
    Y = np.array(Y)
    model = np.polyfit(X, Y, 1)

    # we do one iteration of sigma clipping
    X1 = []
    Xg1 = []
    Y1 = []
    for i in range(len(X)):
        ypred = model[0]*X[i] + model[1]
        if np.abs(Y[i] - ypred) < dmag_filter:
            X1.append(X[i])
            Xg1.append(Xg[i])
            Y1.append(Y[i])

    X1 = np.array(X1)
    Xg1 = np.array(Xg1)
    Y1 = np.array(Y1)
    model = np.polyfit(X1, Y1, 1)
    
    # with respect to this model
    ocam_prediced_mag = []
    for i in range(len(x)):
        ocam_prediced_mag = np.append(ocam_prediced_mag, model[0]*x[i] + model[1])
    
    # filter nan and inf values
    good_values1 = ~np.isnan(y)
    good_values2 = ~np.isinf(y)
    good_values = good_values1 & good_values2
    
    sigma = np.std(y[good_values] - ocam_prediced_mag[good_values])
    
    slope = model[0]
    zp = model[1]
    return zp, slope, sigma

mag_min = 13.5
mag_max = 17.0
dmag_filter = 0.2
zp, slope, sigma = getZP(sm_r_SDSS, sm_g_SDSS, ocam_r_SDSS, mag_min, mag_max, dmag_filter)
print(zp, slope, sigma)

# we plot the ocam vs skymapper relation
# figsize=(8, 4)
fig = plt.figure(figsize=(9.5, 6))

gs = gridspec.GridSpec(nrows=4, ncols=5, wspace=0.0) # width_ratios=[1,1,1,1,1])
ax1 = fig.add_subplot(gs[:-2,:-3])
ax2 = fig.add_subplot(gs[-2, :-3], sharex=ax1)
ax3 = fig.add_subplot(gs[:-1,  -3:])
ax4 = fig.add_subplot(gs[-1,:])


#plt.figure(figsize = (10,10))
#ax1.set_aspect('equal')
ax1.scatter(sm_r_SDSS, ocam_r_SDSS,  color='black', s=2)
ax1.set_xlim(7.0, 17.0)

# delta zp vs mag
ax2.scatter(sm_r_SDSS, ocam_r_SDSS - sm_r_SDSS,  color='black', s=2)
ax2.set_ylim(zp-10.0*sigma, zp+10.0*sigma)
ax2.axhline(zp, ls=':')

# abd we plot
# read wcs from image keywords
wcs = WCS(reduced_dir+'coadd.fits')  
norm = simple_norm(data, 'sqrt', percent=99)
ax3.imshow(data, norm=norm)
aperture.to_pixel(wcs).plot(color='white', lw=1)

plt.setp(ax3.get_xticklabels(), visible=False)
plt.setp(ax3.get_yticklabels(), visible=False)

# delta zp vs color
color = sm_g_SDSS - sm_r_SDSS
select = sm_r_SDSS > 13.5

# the slope of 0.994 gives a bias at fainter magnitudes
# that translate into an offset. If we set slope to 1
# for this plot we get an offset 0.1 mags.
#
ax4.scatter(color[select], ocam_r_SDSS[select] - slope*sm_r_SDSS[select],  color='black', s=2)
ax4.set_ylim(zp-3.0*sigma, zp+3.0*sigma)
ax4.set_xlim(-0.5, 2.5)
ax4.axhline(zp, ls=':')
#plt.setp([ax1, ax2], title='ZP Conversion')

#plt.ylim(7, mlim)
plt.show()
print(color)
#print(phot_table)

In [ ]:
# Plot zeropoint versus magnitude
x = sm_r_SDSS
xg = sm_g_SDSS
y = ocam_r_SDSS

plt.figure(figsize = (10,3))
#plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(xg - x, y - (slope*x+zp),  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper (g-r)_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

#plt.plot(X1, Y1 - (model[0]*X1+model[1]), color='red',linewidth=2)
plt.show()

# Plot zeropoint versus color
plt.figure(figsize = (10,3))
plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(x, y - (slope*x+zp),  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper r_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

plt.show()

# y = model[0]*x + model[1]
print(slope, zp)
mag_zeropoint = -zp
print('mag_zeropoint: %6.3f'%(mag_zeropoint))
fits.setval(reduced_dir+'coadd.fits', 'MAGZP', value=mag_zeropoint, comment='Magnitude zp from SkyMapper', before='EQUINOX', after=None, savecomment=False)

# read wcs info to get plate scale
wcs_keys = getKeyListVal(reduced_dir+'coadd.fits', ['CD1_1', 'CD1_2', 'CD2_1', 'CD2_2'])
cd1_1 = wcs_keys['CD1_1']
cd1_2 = wcs_keys['CD1_2']
cd2_1 = wcs_keys['CD2_1']
cd2_2 = wcs_keys['CD2_2']
plate_scale = np.sqrt(-(cd1_1*cd2_2 - cd1_2*cd2_1))*3600.0
print('plate_scale: ', plate_scale)

# now we do some statistics to get the sky
from numpy import errstate,isneginf,array
sky_median = np.nanmedian(data)
sky_flux_std = np.nanstd(data)

# we calculate the background std. deviation sigma-clipping once
sky_flux_std = np.std(data[data < sky_median+5.0*sky_flux_std])


In [ ]:
# now we do some statistics to get the sky
# But we should consider that during the
# swarp process we lost the sky. Therefore,
# we will retrieve the sky from 
# the OMEGA.YYYY-MM-DDThh:mm:ss.sss_pre_wcs.fits files
# We loop through every file
# retrieve the median sky values from each chip
# create the median and std deviation per frame
# add these keywords to the final frame
# at the end we add the mean over all medians
# to the coadd.fits file.
ext_name_list = ["ESO_CCD_#%2d"%(i) for i in range(65,97)]
  

list_to_process = glob.glob(reduced_dir+'OMEGA.*_pre/OMEGA.*_pre_wcs.fits')

sky_mean_list = []
sky_std_list = []
if len(list_to_process) > 0:
    i = 0
    for file_name in list_to_process:
        i += 1
        if i <= len(list_to_process):
            hdu_list = fits.open(file_name)
            print('Finding sky %i/%i: %s'%(i, len(list_to_process), os.path.basename(file_name),))
            median_list = []
            for extname in ext_name_list:
                hdr = fits.getheader(file_name, extname=extname)
                data = fits.getdata(file_name, extname=extname)
                median_list.append(np.nanmedian(data))
            median_list = np.array(median_list)
            
            sky_mean = np.nanmean(median_list)
            sky_mean_list.append(sky_mean)
            
            sky_std = np.nanstd(median_list)            
            sky_std_list.append(sky_std)
            print(os.path.basename(file_name), sky_mean, sky_std,)
            
            #hdu_list.info()
print('Done.')



In [ ]:
# Now we determine the overall sky level
# to be added to coadd.fits
#
# Actually, we need the convertion factor
# we need to multiply the pixel values so that
# we get a signal in erg/s/cm^-2/Angstrom
# that will also permit us to get magnitudes
# with aperture photometry with the formula
#
# mag = f0 - 2.5 log10(Sum_pix_ij val_ij)
# 
# where f0 is given in the header keyword
#
# HIERARCH MEFPP QC F0_PHYSICAL
#
sky_flux = 'HIERARCH MEFPP QC SKY FLUX'
sky_mag = 'HIERARCH MEFPP QC SKY MAG'
nvalid_skies = len(sky_mean_list)

print(nvalid_skies)
good_sky = (nvalid_skies == len(list_to_process))
print('Good sky?: ', good_sky)
sky_mean_mean = np.nanmean(sky_mean_list)
sky_mean_max = np.nanmax(sky_mean_list)
sky_mean_min = np.nanmax(sky_mean_list)

std_mean_max = np.nanmax(sky_std_list)
std_mean_min = np.nanmin(sky_std_list)

sky_delta_mag = 2.5*sky_flux_std/(sky_mean_mean*np.log(10.0))
print('sky_mag: ', -2.5*np.log10(sky_mean_mean/plate_scale**2) + mag_zeropoint)
print('sky_delta_mag: ', sky_delta_mag)



# Could not make this work. To continue testing change the cell type from Markdown to Code

from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.vo_conesearch import conesearch
from astroquery.vo_conesearch.vos_catalog import VOSDatabase
import warnings

## To explore catalog availability uncomment this file and then go to the cell named explore catalogues...

from astroquery.vo_conesearch.validator import conf as validator_conf
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    registry_db = VOSDatabase.from_registry(
        validator_conf.conesearch_master_list, encoding='binary',
        cache=False)
len(registry_db)    
conesearch.list_catalogs()
## EXPLORE CATALOGS

### the commented lines were used to explore to find the skymapper catalog
#sm_a2_list = registry_db.list_catalogs(pattern='skymapper')
#sm_a2_list
sm_catalog = registry_db.get_catalog('SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1')
print(sm_catalog)
#sm_catname = 'SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1'
#skymapper_url = 'http://vizier.u-strasbg.fr/viz-bin/conesearch/II/358/smss?'
#sm = registry_db.get_catalog_by_url(skymapper_url)
print(sm)

c = SkyCoord.from_name('CP Pupis')
c
sm_catname = 'SkyMapper Southern Sky Survey. DR1.1 (Wolf+, 2018) 1'
result = conesearch.conesearch(c, 0.1 * u.degree, catalog_db=sm_catname)